## Task B: Multi-label Criteria Classification

 the goal for Task B is to predict up to four criteria labels for each tweet in the `val` file. I treated this as a multi-label text classification problem and build an ensemble of three models: MARBERT, AraBERT-Twitter, and a TF‑IDF + Logistic Regression baseline.


In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
import re


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/politeness_Impolite_train_TaskBB.csv")
val_df = pd.read_csv("/content/drive/MyDrive/politeness_Impolite_val_TaskBB.csv")


df.head()

### Identify text and criteria columns

I identified the text column in the train and validation data, then collect all non-empty values from the four criteria columns into a single Series. This lets us inspect and normalize the label space used across `criteria 1–4`.


In [ ]:
# unnamed columns(the text)
text_col_train = df.columns[0]
text_col_val   = val_df.columns[0]

CRIT_COLS = ["criteria 1", "criteria 2", "criteria 3", "criteria 4"]

# stack all non-null criteria into one long Series
all_criteria = (
    df[CRIT_COLS]
    .values
    .ravel()
)

# convert to Series, drop NaN/empty
all_criteria = pd.Series(all_criteria)
all_criteria = all_criteria[all_criteria.notna()]
all_criteria = all_criteria[all_criteria.astype(str).str.strip() != ""]


### Defining canonical label set and mapping

The raw criteria columns contain several string variants for the same conceptual label. To make training consistent, i first define a fixed list of canonical labels in `CATEGORIES` containing the 9 given categories in the task. Then i create a `LABEL_MAPPING` dictionary that maps noisy or alternative text variants from the annotation into these canonical category names

In [ ]:
counts = all_criteria.value_counts().sort_values(ascending=False)
print(counts)

In [ ]:
#mapping into the 9 categories:

CATEGORIES = [
    "Criticism",
    "Insult",
    "Respect",
    "Prayers",
    "Greetings",
    "Hospitality",
    "Gratitude",
    "Admiration / Love",
    "Racism / Discrimination",
]

LABEL_MAPPING = {
    "Insult": "Insult",
    "Disparagement": "Insult",
    "Verbal violence - عنف لفظي": "Insult",
    "Threat - تهديد": "Insult",
    "Sarcasm - تهكم": "Insult",

    "Criticism": "Criticism",
    "Criticism - انتقاد": "Criticism",
    "انتقاد - Criticism": "Criticism",
    "Accusation": "Criticism",

    "Respect": "Respect",
    "Asking for permission": "Respect",
    "Asking for permission - طلب الإذن": "Respect",
    "Asking for permission -  طلب الإذن و التودد": "Respect",
    "Asking for permission - طلب الإذن و التودد": "Respect",
    "Asking for permission -  طلب الإذن": "Respect",
    "اعتذار - Excuse": "Respect",

    "Greetings - التحية": "Greetings",

    "Gratitude": "Gratitude",
    "Gratitude - الامتنان": "Gratitude",
    "Gratitude - الامتنان و الشكر": "Gratitude",
    "Thanks & gratitude - الشكر و الامتنان": "Gratitude",
    "Congratulations - تهنئة": "Gratitude",
    "Felicitation - التهنئة": "Gratitude",
    "التهنئة": "Gratitude",
    "Prayers, Felicitation - التهنئة": "Gratitude",

    "Admiration - الإعجاب": "Admiration / Love",
    "Love - الحب": "Admiration / Love",
    "Appreciation & love - الإعجاب و الحب": "Admiration / Love",
    "Appreciation & Love - الإعجاب و الحب": "Admiration / Love",
    "Admiration & Love - الإعجاب و الحب": "Admiration / Love",
    "Admiration & Love - الإعجاب و الحب, Gratitude - الامتنان و الشكر": "Admiration / Love",
    "Admiration & Love - الإعجاب و الحب, Prayers": "Admiration / Love",


    "Hospitality & generosity - الضيافة و الكرم": "Hospitality",


    "Prayers": "Prayers",

    "Racism / Discrimination": "Racism / Discrimination",
    "Racism & discrimination - عنصرية و تمييز": "Racism / Discrimination",
    "Discrimination, secterian, & discrimination - عنصرية، تحزب و تمييز": "Racism / Discrimination",
    "& discrimination - عنصرية، تحزب و تمييز": "Racism / Discrimination"
}

all_criteria_mapped = all_criteria.astype(str).str.strip().map(
    lambda x: LABEL_MAPPING.get(x, x)
)

### Map raw criteria values to canonical label names

I define a helper `parse_row_labels` that reads all four criteria columns for a row, applies `LABEL_MAPPING` to harmonize different string variants, and keeps only labels that belong to our final `CATEGORIES` list.


In [ ]:
def parse_row_labels(row):
    labs = []
    for col in CRIT_COLS:
        if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
            raw = str(row[col]).strip()
            mapped = LABEL_MAPPING.get(raw, raw)
            if mapped in CATEGORIES:
                labs.append(mapped)
    return labs


df["labels_list"] = df.apply(parse_row_labels, axis=1)
val_df["labels_list"]   = val_df.apply(parse_row_labels, axis=1)

### Visualize label distribution across criteria

I computed the frequency of each mapped category and plot a bar chart to inspect class imbalance. Which helped us identify the minority labels that require oversampling.


In [ ]:
counts = all_criteria_mapped.value_counts().sort_values(ascending=False)
print(counts)

As shown above, the dataset in imbalanced, so I will apply oversampling to the minority classes.

In [ ]:

# minority classes
minority = ["Hospitality", "Racism / Discrimination", "Greetings"]

def has_minority_label(labels):
    return any(l in minority for l in labels)

minority_df = df[df["labels_list"].apply(has_minority_label)]
print("Minority rows:", len(minority_df))

k = 3  # oversampling factor
df_bal = pd.concat([df, minority_df] * k, ignore_index=True)
print("Original train size:", len(df))
print("Balanced train size:", len(df_bal))

In [ ]:
all_labels_bal = df_bal["labels_list"].explode()

# Count per category (keep fixed order)
counts_bal = all_labels_bal.value_counts().reindex(CATEGORIES, fill_value=0)

print(counts_bal)

In [ ]:


# Ensure both Series are in the CATEGORIES order and handle potential missing values
combined_counts = pd.DataFrame({
    'Before Oversampling': counts.reindex(CATEGORIES, fill_value=0),
    'After Oversampling': counts_bal.reindex(CATEGORIES, fill_value=0)
})

plt.figure(figsize=(14, 7))
combined_counts.plot(kind='bar', figsize=(14, 7), color=['skyblue', 'orange'])
plt.title('Category Distribution Before and After Oversampling')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Oversampling Status')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### Convert label lists to multi-hot matrices

I use `MultiLabelBinarizer` with the fixed `CATEGORIES` to transform `labels_list` into multi-hot matrices.

In [ ]:
#rebuilding the traininf df

mlb = MultiLabelBinarizer(classes=CATEGORIES)
Y_train_bal = mlb.fit_transform(df_bal["labels_list"])
Y_val = mlb.transform(val_df["labels_list"])

print("Balanced per-class counts:")
for cat, cnt in zip(CATEGORIES, Y_train_bal.sum(axis=0)):
    print(f"{cat:25s}: {int(cnt)}")


print("Y_train_bal shape:", Y_train_bal.shape)
print("Y_val shape:", Y_val.shape)

### Train TF-IDF + Logistic Regression baseline



In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
)

X_train_tfidf = tfidf.fit_transform(df_bal[text_col_train])
X_val_tfidf   = tfidf.transform(val_df[text_col_val])

baseline_clf = OneVsRestClassifier(
    LogisticRegression(max_iter=2000, n_jobs=-1)
)
baseline_clf.fit(X_train_tfidf, Y_train_bal)

val_pred_baseline = baseline_clf.predict(X_val_tfidf)
f1_macro_base = f1_score(Y_val, val_pred_baseline, average="macro", zero_division=0)
print("Baseline TF-IDF + LR macro F1:", f1_macro_base)

Baseline TF-IDF + LR macro F1: 0.3665230243461161


In [ ]:
from sklearn.metrics import classification_report

print("TF-IDF + Logistic Regression Baseline - Per-category F1 Scores:")
print(classification_report(
    Y_val,
    val_pred_baseline,
    target_names=CATEGORIES,
    zero_division=0,
    digits=4,
))


TF-IDF + Logistic Regression Baseline - Per-category F1 Scores:
                         precision    recall  f1-score   support

              Criticism     0.0000    0.0000    0.0000        30
                 Insult     0.9167    0.4459    0.6000        74
                Respect     0.9383    0.6909    0.7958       110
                Prayers     0.8974    0.5932    0.7143        59
              Greetings     1.0000    0.1875    0.3158        16
            Hospitality     0.0000    0.0000    0.0000         1
              Gratitude     1.0000    0.3448    0.5128        29
      Admiration / Love     0.8182    0.2308    0.3600        39
Racism / Discrimination     0.0000    0.0000    0.0000         1

              micro avg     0.9222    0.4624    0.6160       359
              macro avg     0.6190    0.2770    0.3665       359
           weighted avg     0.8382    0.4624    0.5795       359
            samples avg     0.5103    0.4602    0.4747       359



## MARBERT


In [ ]:
MODEL_NAME_a = "UBC-NLP/MARBERT"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


tokenizer_a = AutoTokenizer.from_pretrained(MODEL_NAME_a)


model_a = AutoModelForSequenceClassification.from_pretrained(
 MODEL_NAME_a,
 num_labels=len(CATEGORIES),
 problem_type="multi_label_classification",
)
model_a.to(DEVICE)


## AraBERT



In [ ]:
MODEL_NAME_b = "aubmindlab/bert-base-arabertv02-twitter"
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_NAME_b)

model_b = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_b,
    num_labels=len(CATEGORIES),
    problem_type="multi_label_classification",
)
model_b.to(DEVICE)


### Normalize Arabic text for BERT models

I define `preprocess_bert` to normalize Arabic characters and remove extra whitespace, then apply it to build a `text_clean` column for both the balanced train set and validation. These cleaned texts are used as inputs to MARBERT and AraBERT.


In [ ]:
def preprocess_bert(text):
  if pd.isna(text):
    return ""

   # Normalize Arabic characters
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'ء', text)
    text = re.sub(r'ئ', 'ء', text)

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text

df_bal["text_clean"] = df_bal[text_col_train].apply(preprocess_bert)
val_df["text_clean"]   = val_df[text_col_val].apply(preprocess_bert)

### Create PyTorch Dataset for MARBERT and AraBERT

I implement `TheDataset` to tokenize each cleaned text, return `input_ids`, `attention_mask`, and the multi-hot `labels`. Using this, I build separate train/validation datasets for MARBERT and AraBERT with their corresponding tokenizers.


In [ ]:
class TheDataset (Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float),
        }


#for marbert
train_ds_a = TheDataset(df_bal["text_clean"], Y_train_bal, tokenizer_a, max_length=128)
val_ds_a   = TheDataset(val_df["text_clean"],   Y_val,   tokenizer_a, max_length=128)

#for arabert
train_ds_b = TheDataset(df_bal["text_clean"], Y_train_bal, tokenizer_b, max_length=128)
val_ds_b   = TheDataset(val_df["text_clean"],   Y_val,   tokenizer_b, max_length=128)


### Define validation metric: macro F1




In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"f1": macro_f1}



### Use Focal Loss for multi-label training
I replace the default loss with Focal Loss, which can help with label imbalance in multi-label settings.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce = nn.BCEWithLogitsLoss(reduction="none")(inputs, targets)
        pt = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

class FocalLossTrainer(Trainer):
    def __init__(self, *args, use_focal_loss=True, focal_alpha=0.25, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_focal_loss = use_focal_loss
        if use_focal_loss:
            self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

    def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.use_focal_loss:
            loss = self.focal_loss(logits, labels)
        else:
            loss = nn.BCEWithLogitsLoss()(logits, labels)
        return (loss, outputs) if return_outputs else loss


### Configure training hyperparameters for MARBERT and AraBERT


In [ ]:

#  for MARBERT
training_args_a = TrainingArguments(
    output_dir="./marbert_taskB",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)


# for AraBERT
training_args_b = TrainingArguments(
    output_dir="./arabert_twitter_taskB",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs_arabert_tw",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)



### Training

In [ ]:
#MARBERT
trainer_a = FocalLossTrainer(
    model=model_a,
    args=training_args_a,
    train_dataset=train_ds_a,
    eval_dataset=val_ds_a,
    compute_metrics=compute_metrics,
    use_focal_loss=True,
    focal_alpha=0.25,
    focal_gamma=2.0,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_a.train()

In [ ]:
#AraBERT
trainer_b = FocalLossTrainer(
    model=model_b,
    args=training_args_b,
    train_dataset=train_ds_b,
    eval_dataset=val_ds_b,
    compute_metrics=compute_metrics,
    use_focal_loss=True,
    focal_alpha=0.25,
    focal_gamma=2.0,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_b.train()

## Saving the trained models

In [ ]:
# MARBERT
trainer_a.save_model("./marbert_taskB_final")
tokenizer_a.save_pretrained("./marbert_taskB_final")

# AraBERT-Twitter
trainer_b.save_model("./arabert_twitter_taskB_final")
tokenizer_b.save_pretrained("./arabert_twitter_taskB_final")

### extracting the probabilities on validation

In [ ]:

# MARBERT
pred_a= trainer_a.predict(val_ds_a)
probs_val_a = torch.sigmoid(torch.tensor(pred_a.predictions)).numpy()

#AraBERT
pred_b = trainer_b.predict(val_ds_b)
probs_val_b = torch.sigmoid(torch.tensor(pred_b.predictions)).numpy()


# TF-IDF baseline
probs_val_base = np.zeros_like(probs_val_a)
for i in range(len(CATEGORIES)):
    est = baseline_clf.estimators_[i]  # LogisticRegression for label i
    probs_val_base[:, i] = est.predict_proba(X_val_tfidf)[:, 1]

## Ensemble : *baseline+ MARBERT & AraBERT &*
### Tune a global threshold per transformer model

I define `find_best_thr` to search a single probability threshold to maximizes macro F1. I applied it to MARBERT and AraBERT probabilities.

In [ ]:
def find_best_thr(probs_val, Y_val):
    ths = np.linspace(0.0, 0.9, 91)
    best_thr = 0.0
    best_f1 = 0.0
    for t in ths:
        preds = (probs_val > t).astype(int)
        f1 = f1_score(Y_val, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = t
    return best_thr, best_f1

best_thr_a, best_f1_a = find_best_thr(probs_val_a, Y_val)
best_thr_b, best_f1_b = find_best_thr(probs_val_b, Y_val)


preds_a = (probs_val_a > best_thr_a).astype(int)
preds_b = (probs_val_b > best_thr_b).astype(int)



print(classification_report(
    Y_val,
    preds_a,
    target_names=CATEGORIES,
    zero_division=0,
    digits=4,
))

print(classification_report(
    Y_val,
    preds_b,
    target_names=CATEGORIES,
    zero_division=0,
    digits=4,
))



                         precision    recall  f1-score   support

              Criticism     0.1031    1.0000    0.1869        30
                 Insult     0.2543    1.0000    0.4055        74
                Respect     0.3780    1.0000    0.5486       110
                Prayers     0.2027    1.0000    0.3371        59
              Greetings     0.0550    1.0000    0.1042        16
            Hospitality     0.0034    1.0000    0.0068         1
              Gratitude     0.0997    1.0000    0.1812        29
      Admiration / Love     0.1340    1.0000    0.2364        39
Racism / Discrimination     0.0034    1.0000    0.0068         1

              micro avg     0.1371    1.0000    0.2411       359
              macro avg     0.1371    1.0000    0.2237       359
           weighted avg     0.2353    1.0000    0.3677       359
            samples avg     0.1371    1.0000    0.2372       359

                         precision    recall  f1-score   support

              Critici

### Optimize per-class decision thresholds for the ensemble

`optimize_per_class_thresholds` tunes a separate threshold for each label by scanning a grid between `t_min` and `t_max`. For each label, it selects the threshold that maximizes binary F1 on that label, returning a vector of best thresholds.


In [ ]:
def optimize_per_class_thresholds(scores, Y_true, categories, t_min=0.05, t_max=0.8, step=0.05):

    n_labels = len(categories)
    thresholds = np.zeros(n_labels)

    ts = np.arange(t_min, t_max + 1e-9, step)

    for j in range(n_labels):  # per label
        best_t = t_min
        best_f1 = 0.0
        y_true_j = Y_true[:, j]
        scores_j = scores[:, j]

        for t in ts:
            y_pred_j = (scores_j > t).astype(int)
            f1 = f1_score(y_true_j, y_pred_j, average="binary", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        thresholds[j] = best_t

    return thresholds


### Tune ensemble weights and per-class thresholds

In [ ]:

alphas = [
    (0.40, 0.30, 0.30),
    (0.40, 0.40, 0.20),
    (0.50, 0.30, 0.20),
    (0.30, 0.30, 0.40),
    (0.20, 0.30, 0.50),
    (0.30, 0.20, 0.50)
]


best_combo = None
best_f1 = 0.0
best_thr_vec = None
best_scores = None

for w_a, w_b, w_base in alphas:
    scores = (
        w_a   * probs_val_a +
        w_b  * probs_val_b +
        w_base* probs_val_base
    )

    thr_vec = optimize_per_class_thresholds(
        scores, Y_val, CATEGORIES, t_min=0.02, t_max=0.7, step=0.01
    )

    preds = np.zeros_like(scores)
    for j in range(len(CATEGORIES)):
        preds[:, j] = (scores[:, j] > thr_vec[j]).astype(int)

    f1 = f1_score(Y_val, preds, average="macro", zero_division=0)
    print(f"[M={w_a}, A-TW={w_b}, TFIDF={w_base}] -> macro F1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_combo = (w_a, w_b, w_base)
        best_thr_vec = thr_vec
        best_scores = scores

ensemble_scores = best_scores
print("Best 3-model ensemble:", best_combo, best_f1)




[M=0.4, A-TW=0.3, TFIDF=0.3] -> macro F1=0.6457
[M=0.4, A-TW=0.4, TFIDF=0.2] -> macro F1=0.6339
[M=0.5, A-TW=0.3, TFIDF=0.2] -> macro F1=0.6372
[M=0.3, A-TW=0.3, TFIDF=0.4] -> macro F1=0.6516
[M=0.2, A-TW=0.3, TFIDF=0.5] -> macro F1=0.6504
[M=0.3, A-TW=0.2, TFIDF=0.5] -> macro F1=0.6538
Best 3-model ensemble: (0.3, 0.2, 0.5) 0.6537508224028716


In [ ]:
from sklearn.metrics import classification_report

print("Overall Ensemble Model - Per-category F1 Scores:")
print(classification_report(
    Y_val,
    binary_preds_ens,
    target_names=CATEGORIES,
    zero_division=0,
    digits=4,
))


Overall Ensemble Model - Per-category F1 Scores:
                         precision    recall  f1-score   support

              Criticism     0.4167    0.3333    0.3704        30
                 Insult     0.7381    0.8378    0.7848        74
                Respect     0.8824    0.8182    0.8491       110
                Prayers     0.8596    0.8305    0.8448        59
              Greetings     1.0000    0.5625    0.7200        16
            Hospitality     1.0000    1.0000    1.0000         1
              Gratitude     0.7600    0.6552    0.7037        29
      Admiration / Love     0.5294    0.6923    0.6000        39
Racism / Discrimination     0.0055    1.0000    0.0110         1

              micro avg     0.5019    0.7465    0.6002       359
              macro avg     0.6880    0.7478    0.6538       359
           weighted avg     0.7649    0.7465    0.7487       359
            samples avg     0.5326    0.7483    0.5965       359




### Evaluate final ensemble and create the submission CSV

In [ ]:

MAX_LABELS = 4

def probs_to_labels_row(scores_row, categories, thr_vec, max_labels=4):
    idx_sorted = np.argsort(-scores_row)  # descending by score
    labels = []
    for idx in idx_sorted:
        if scores_row[idx] < thr_vec[idx]:
            continue
        labels.append(categories[idx])
        if len(labels) == max_labels:
            break
    return labels

def labels_to_cols(labels, max_cols=4):
    cols = [""] * max_cols
    for i, lab in enumerate(labels[:max_cols]):
        cols[i] = lab
    return cols


binary_preds_ens = np.zeros_like(ensemble_scores)
for j in range(len(CATEGORIES)):
    binary_preds_ens[:, j] = (ensemble_scores[:, j] > best_thr_vec[j]).astype(int)

val_f1 = f1_score(Y_val, binary_preds_ens, average="macro", zero_division=0)
print(f"Validation macro F1 (final ensemble): {val_f1:.4f}")

rows = []
for i in range(len(val_df)):
    labels = probs_to_labels_row(
        ensemble_scores[i],
        CATEGORIES,
        thr_vec=best_thr_vec,
        max_labels=MAX_LABELS,
    )
    c1, c2, c3, c4 = labels_to_cols(labels, max_cols=4)
    rows.append([c1, c2, c3, c4])

subtaskB = pd.DataFrame(rows, columns=["criteria 1", "criteria 2", "criteria 3", "criteria 4"])
subtaskB.to_csv("subtaskB.csv", index=False)
subtaskB.head()


